# Pediatric Chest X-Ray Pneumonia Classification
DenseNet-121 transfer learning project. This notebook is an educational prototype, not a medical diagnostic tool.

## 1. Install and import dependencies

In [ ]:
!pip -q install kaggle scikit-learn matplotlib seaborn
import os, zipfile, random
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Download the Kaggle dataset
Add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Colab Secrets before running. Do not commit credentials.

In [ ]:
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content --unzip
DATA_DIR = Path('/content/chest_xray')
assert all((DATA_DIR/split/cls).exists() for split in ['train','val','test'] for cls in ['NORMAL','PNEUMONIA'])

## 3. Prepare image datasets

In [ ]:
mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]
train_tf = transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(), transforms.RandomRotation(10), transforms.ToTensor(), transforms.Normalize(mean,std)])
eval_tf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), transforms.Normalize(mean,std)])
train_ds = datasets.ImageFolder(DATA_DIR/'train', transform=train_tf)
val_ds = datasets.ImageFolder(DATA_DIR/'val', transform=eval_tf)
test_ds = datasets.ImageFolder(DATA_DIR/'test', transform=eval_tf)
assert train_ds.class_to_idx == val_ds.class_to_idx == test_ds.class_to_idx
train_dl = DataLoader(train_ds,batch_size=32,shuffle=True,num_workers=2)
val_dl = DataLoader(val_ds,batch_size=32,shuffle=False,num_workers=2)
test_dl = DataLoader(test_ds,batch_size=32,shuffle=False,num_workers=2)
print(train_ds.class_to_idx)

## 4. Build DenseNet-121

In [ ]:
weights = models.DenseNet121_Weights.DEFAULT
model = models.densenet121(weights=weights)
model.classifier = nn.Linear(model.classifier.in_features, 2)
model = model.to(device)
counts = torch.bincount(torch.tensor(train_ds.targets), minlength=2).float()
class_weights = (counts.sum()/(2*counts)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

## 5. Train and select the best model using validation accuracy

In [ ]:
best_val = 0.0
for epoch in range(5):
    model.train(); total=correct=loss_sum=0
    for x,y in train_dl:
        x,y=x.to(device),y.to(device); optimizer.zero_grad()
        logits=model(x); loss=criterion(logits,y); loss.backward(); optimizer.step()
        loss_sum += loss.item()*y.size(0); correct += (logits.argmax(1)==y).sum().item(); total += y.size(0)
    model.eval(); vcorrect=vtotal=0
    with torch.no_grad():
        for x,y in val_dl:
            x,y=x.to(device),y.to(device); pred=model(x).argmax(1)
            vcorrect += (pred==y).sum().item(); vtotal += y.size(0)
    val_acc=vcorrect/vtotal
    print(f'Epoch {epoch+1}: train loss={loss_sum/total:.4f}, train acc={correct/total:.3f}, val acc={val_acc:.3f}')
    if val_acc > best_val:
        best_val=val_acc; torch.save(model.state_dict(),'best_pneumonia_densenet.pth')
model.load_state_dict(torch.load('best_pneumonia_densenet.pth',map_location=device,weights_only=True))

## 6. Final held-out test evaluation

In [ ]:
model.eval(); y_true=[]; y_pred=[]; y_prob=[]
with torch.no_grad():
    for x,y in test_dl:
        probs=torch.softmax(model(x.to(device)),dim=1).cpu().numpy()
        y_true.extend(y.numpy()); y_pred.extend(probs.argmax(1)); y_prob.extend(probs[:,1])
print(classification_report(y_true,y_pred,target_names=train_ds.classes))
print('Confusion matrix:\n',confusion_matrix(y_true,y_pred))
print('ROC-AUC:',roc_auc_score(y_true,y_prob))
torch.save(model.state_dict(),'best_pneumonia_densenet.pth')

## Limitations and responsible use
This dataset-level experiment is not clinical validation. Do not use predictions for patient care. Check for patient-level leakage, evaluate external hospitals, and assess calibration before making any research claims.